In [ ]:
import os
import string
from datasets import load_dataset
from mlx_tune import FastLanguageModel, SFTTrainer, SFTConfig
from transformers import TrainingArguments, set_seed
from mlx_lm import generate
from mlx_tune.chat_templates import get_chat_template, standardize_sharegpt

In [ ]:
set_seed(42)

In [ ]:
dataset = load_dataset("json", data_files="../../data/processed/haikus_dataset.json", split="train")

In [ ]:
dataset

In [ ]:
dataset = standardize_sharegpt(dataset)

In [ ]:
max_seq_length = 2048

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "google/gemma-3-1b-it",
    # model_name = "google/gemma-3-1b-pt",
    max_seq_length = max_seq_length,
    load_in_4bit = False
)

In [ ]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = []
    for convo in convos:
        formatted = tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False
        )
        
        
        texts.append(formatted)
    return {"text": texts}

In [ ]:
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"✅ Preprocessing complete. Samples formatted: {len(dataset)}")

In [ ]:
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

In [ ]:
print(f"Training on: {len(train_dataset)} | Evaluating on: {len(eval_dataset)}")

In [ ]:
try:
    from gemmaiku import get_syllable_count_for_line as count_syllables_line
except ModuleNotFoundError:
    import sys
    import os
    sys.path.append(os.path.abspath("../../src"))
    from gemmaiku import get_syllable_count_for_line as count_syllables_line


In [ ]:
def evaluate_haiku(text):
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) != 3:
        return [0, 0, 0], False
    counts = [count_syllables_line(l) for l in lines]
    is_perfect = (counts == [5, 7, 5])
    return counts, is_perfect

In [ ]:
from mlx_lm.sample_utils import make_sampler
import pandas as pd

def run_internal_eval_detailed(model, tokenizer, test_samples, temp=0.0):
    results_list = []
    sampler = make_sampler(temp=temp)

    for sample in test_samples:
        user_prompt = sample['messages'][0]['content']

        eval_messages = [
             {"role": "user", "content": user_prompt}
        ]
        
        prompt = tokenizer.apply_chat_template(
            eval_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        response = generate(model, tokenizer, prompt=prompt, max_tokens=64, sampler=sampler)

        haiku = response.split("<end_of_turn>")[0].strip()

        counts, is_perfect = evaluate_haiku(haiku)
        total_syllables = sum(counts)

        results_list.append({
            "topic": user_prompt,
            "counts": counts,
            "total": total_syllables,
            "is_perfect": is_perfect,
            "error_dist": abs(total_syllables - 17)
        })
        
    return pd.DataFrame(results_list)


In [ ]:
# for name, module in model.named_modules():
#     if "proj" in name:
#         print(name)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", 
                      "self_attn.o_proj", "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj"],
    lora_alpha = 32,
    lora_dropout = 0, 
    bias = "none",    
)

In [ ]:
training_config = SFTConfig(
    output_dir = "mlx_outputs",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    max_steps = 2000,             
    max_length = max_seq_length,
    learning_rate = 2e-4,
    logging_steps = 15,
    lr_scheduler_type = "constant",
    dataset_text_field = "text",       
)

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,      
    train_dataset = train_dataset,
    args = training_config,            
)

In [ ]:
print("--- TRAINER INPUT CHECK ---")
print(train_dataset[0]['text'][:100])

In [ ]:
print("🍏 Launching Metal-accelerated fine-tuning loop on your M1 GPU...")
trainer.train()
print("✅ Training sequence completely run!")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def visualize_results(df):
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)

    sns.histplot(df['total'], bins=10, color='skyblue', kde=True)
    plt.axvline(17, color='red', linestyle='--', label='Perfect (17)')
    plt.title("Syllable Count Distribution")
    plt.legend()

    plt.subplot(1, 2, 2)
    
    # Convert to boolean series safely (handles bool, string, and repeat runs)
    is_perfect_bool = df['is_perfect'].isin([True, 'True', 'Perfect'])
    accuracy = is_perfect_bool.mean() * 100
    
    plot_df = df.copy()
    plot_df['is_perfect'] = is_perfect_bool.map({False: "Failure", True: "Perfect"})
    
    sns.countplot(x='is_perfect', data=plot_df, hue='is_perfect', palette={'Perfect': '#2ebd59', 'Failure': '#df3e3e'}, legend=False)
    plt.title(f"Accuracy: {accuracy:.1f}%")

    plt.tight_layout()
    plt.show()

In [ ]:
eval_df = run_internal_eval_detailed(model, tokenizer, eval_dataset)

visualize_results(eval_df)

In [ ]:
eval_df["counts"]

In [ ]:
!pip install mlx-lm

In [ ]:
!python -m mlx_lm fuse --model google/gemma-3-1b-it --adapter-path mlx_outputs/adapters --save-path ../../models/gemmaiku-1b

In [ ]:
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
from mlx_lm import load, generate

model, tokenizer = load("../../models/gemmaiku-1b")

while True:
    user_input = input("Topic: ")
    prompt = f"<start_of_turn>user\n{user_input}<end_of_turn>\n<start_of_turn>model\n"

    response = generate(model, tokenizer, prompt=prompt, max_tokens=64)

    print(response.split("<end_of_turn>")[0].strip())